# Assignment 5: Data Analytics II
## Logistic Regression for Classification using Social Network Ads Dataset

**Name:** ___________ | **Class:** T.E. | **Roll No:** ___________

## Problem Statement
To implement Logistic Regression to classify whether a user will purchase a product based on features like age and estimated salary, and to compute evaluation metrics using a confusion matrix.

## Theory
**Logistic Regression:** Supervised ML algorithm for binary classification. Uses sigmoid function to output probabilities between 0 and 1. Classification threshold = 0.5.

**Sigmoid Function:** p = 1 / (1 + e^(-z)), where z = b0 + b1*x1 + ...

**Confusion Matrix:**
|                     | Predicted: 0 | Predicted: 1 |
|---------------------|:------------:|:------------:|
| **Actual: 0**       | TN           | FP           |
| **Actual: 1**       | FN           | TP           |

**Metrics:**
- Accuracy = (TP+TN) / Total
- Precision = TP / (TP+FP)
- Recall = TP / (TP+FN)

In [ ]:
# Step 1: Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score

print("Libraries imported successfully!")

In [ ]:
# Step 2: Load Social Network Ads Dataset
# Load from common source
df = pd.read_csv('https://raw.githubusercontent.com/neha-93/datasets/main/Social_Network_Ads.csv')

# Remove User ID column (not a feature)
if 'User ID' in df.columns:
    df.drop('User ID', axis=1, inplace=True)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head(10)

In [ ]:
# Step 3: Convert Categorical Variable (Gender) into Numeric
le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])  # Male=1, Female=0

print(f"Gender mapping: {dict(zip(le.classes_, range(len(le.classes_))))}")
print(f"
Unique Gender values after encoding: {df['Gender'].unique()}")
df.head()

In [ ]:
# Step 4: Select Features (X) and Target (y)
# Features: Age and EstimatedSalary
X = df[['Age', 'EstimatedSalary']].values
y = df['Purchased'].values

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Target distribution:
  Not Purchased (0): {(y==0).sum()}")
print(f"  Purchased (1):     {(y==1).sum()}")

In [ ]:
# Step 5: Split Dataset into Training and Testing Sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set:  {X_test.shape[0]} samples")

In [ ]:
# Step 6: Create and Train Logistic Regression Model
model = LogisticRegression(random_state=0)
model.fit(X_train, y_train)

print("Logistic Regression model trained successfully!")
print(f"Intercept: {model.intercept_[0]:.4f}")
print(f"Coefficients: Age={model.coef_[0][0]:.4f}, Salary={model.coef_[0][1]:.4f}")

In [ ]:
# Step 7 & 8: Predict Using Test Data
y_pred = model.predict(X_test)

# Compare first 15 predictions
comparison = pd.DataFrame({
    'Age': X_test[:15, 0].astype(int),
    'Salary': X_test[:15, 1].astype(int),
    'Actual': y_test[:15],
    'Predicted': y_pred[:15]
})
print("First 15 predictions vs actual:")
comparison

In [ ]:
# Step 9: Generate Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()

print("CONFUSION MATRIX:")
print("=" * 50)
print(f"{'':15} {'Predicted:0':>12} {'Predicted:1':>12}")
print(f"{'Actual:0':15} {TN:>12} {FP:>12}")
print(f"{'Actual:1':15} {FN:>12} {TP:>12}")
print("=" * 50)

print(f"
  True Negatives (TN):  {TN}")
print(f"  False Positives (FP): {FP}")
print(f"  False Negatives (FN): {FN}")
print(f"  True Positives (TP):  {TP}")

In [ ]:
# Step 10: Calculate Evaluation Metrics
accuracy = accuracy_score(y_test, y_pred)
error_rate = 1 - accuracy
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

# Manual calculation from confusion matrix
total = TN + FP + FN + TP
acc_manual = (TP + TN) / total
err_manual = 1 - acc_manual
prec_manual = TP / (TP + FP) if (TP + FP) > 0 else 0
rec_manual = TP / (TP + FN) if (TP + FN) > 0 else 0

print("EVALUATION METRICS:")
print("=" * 40)
print(f"  Accuracy:   {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"  Error Rate: {error_rate:.4f} ({error_rate*100:.2f}%)")
print(f"  Precision:  {precision:.4f}")
print(f"  Recall:     {recall:.4f}")

In [ ]:
# Visualize: Training set and Test set results
from matplotlib.colors import ListedColormap

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (X_set, y_set, title) in enumerate([
    (X_train, y_train, 'Logistic Regression - Training Set'),
    (X_test, y_test, 'Logistic Regression - Test Set')
]):
    X1, X2 = np.meshgrid(
        np.arange(X_set[:, 0].min()-1, X_set[:, 0].max()+1, 0.5),
        np.arange(X_set[:, 1].min()-1, X_set[:, 1].max()+1, 500)
    )
    Z = model.predict(np.c_[X1.ravel(), X2.ravel()]).reshape(X1.shape)
    
    axes[idx].contourf(X1, X2, Z, alpha=0.3, cmap=ListedColormap(['red', 'green']))
    
    for label, color in [(0, 'red'), (1, 'green')]:
        mask = y_set == label
        axes[idx].scatter(X_set[mask, 0], X_set[mask, 1], c=color, label=f'Class {label}', edgecolor='k', s=25)
    
    axes[idx].set_xlabel('Age')
    axes[idx].set_ylabel('Estimated Salary')
    axes[idx].set_title(title)
    axes[idx].legend()

plt.tight_layout()
plt.show()

## Dataset Description
- **Dataset:** Social Network Ads
- **Records:** ~400 users
- **Features:** User ID, Gender, Age, EstimatedSalary
- **Target:** Purchased (0 = No, 1 = Yes)

## Conclusion
Logistic Regression model successfully classified user purchase behavior. Confusion matrix and derived metrics (Accuracy, Precision, Recall) evaluate model performance. The decision boundary visualization shows how Age and EstimatedSalary separate purchasing vs non-purchasing users.